In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
df_emp = spark.read.format('csv').option('header', True).load('/Volumes/workspace/default/datasets/emp_data_ABC/emp_ABC_update4.CSV')
df_emp.display()

In [0]:
%sql
select * from workspace.default. emp_abc_type_2

In [0]:
df_emp.createOrReplaceTempView('emp_view')

In [0]:
%sql
MERGE into emp_abc1
using emp_view
on emp_abc1.emp_id = emp_view.emp_id
when MATCHED AND emp_view.emp_sal != emp_abc1.emp_sal 
THEN UPDATE
SET
emp_abc1.emp_sal=emp_view.emp_sal,
emp_abc1.year = emp_view.year,
emp_abc1.lst_updt_ts = current_timestamp()
when NOT MATCHED THEN
INSERT (emp_id,emp_name,emp_sal,year,create_ts,lst_updt_ts)
VALUES(emp_view.emp_id,emp_view.emp_name,emp_view.emp_sal,emp_view.year,current_timestamp(),current_timestamp())

In [0]:
%sql
MERGE INTO workspace.default.emp_abc_type_2 tgt
USING emp_view src
on tgt.emp_id =  src.emp_id AND tgt.cur_rec_ind = 'Y'
WHEN MATCHED AND tgt.emp_sal != src.emp_sal
THEN UPDATE 
SET tgt.cur_rec_ind = 'N',
    tgt.lst_updt_ts = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (emp_id,emp_name,emp_sal,year,cur_rec_ind,create_ts,lst_updt_ts)
VALUES(src.emp_id,src.emp_name,src.emp_sal,src.year,'Y',current_timestamp(),current_timestamp())

In [0]:
%sql
select * from emp_abc_type_2